# Node 2 — SubjectAnalysisAgent (technical subjects only)

A bench, not an implementation. Nothing here is imported by `backend/`.

## The design

Two capabilities, not three services:

```
web_search(query, domains=None)
fetch_page(url)
```

One generic search first — the **discovery step**, because we do not yet know what the subject
is. The model reads the results and decides what to do next: fetch first-party pages, run a
targeted search, or declare the name ambiguous. Budget: `MAX_SEARCHES = 3`, `MAX_FETCHES = 5`.

## What carries over from the last run

The previous fan-out build scored **15/15 stable, 13/15 correct**. Both misses were `Statistics`
and `Guitar`, and neither was a reasoning failure — Learn and GitHub occupied four of six document
slots with `netstat`, an XUSB controller subtype, an IIS MIME map and a Qt Git GUI, and the model
correctly reported that those describe several unrelated subjects.

⚠️ **Restricting scope to technical subjects removes both failures on its own.** So routing is
being tested here as a *precision and cost* win, not as a correctness rescue. Do not let the
scoreboard going green be read as proof that routing fixed something.

## Two things being measured, not assumed

- **`confidence`** is in the schema as specified. It is also *instrumented*: §7 compares it on
  correct versus incorrect identifications. A number that reads ~0.95 either way is a dial that
  cannot gate, whatever threshold is put on it. Prior evidence says this is likely — `difficulty`
  returned `intermediate` for 4 of 6 topics including an invented one — so `difficulty` and
  `estimated_hours` are measured for drift in the same cell.
- **Who witnesses the evidence.** The model chooses the strategy; *code* executes every search and
  fetch and records it in a `Trace`. An agent that searched and judged in one turn once reported
  *Rust does not exist* on 1 run in 3, with `sources=0` — indistinguishable from never having
  searched. Here the plan is data, the execution is ours, and the trace is the audit.

⚠️ Planner output selects sources by **index into a numbered list**, never by URL. A URL the model
retypes is a URL it can mistype.


In [28]:
import asyncio
import os
import re
import sys
from collections import Counter
from functools import lru_cache
from itertools import zip_longest
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "backend").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
# Settings resolves env_file=".env" against the CWD, and a notebook's CWD is its own folder.
os.chdir(ROOT)

from backend.config.settings import get_settings

settings = get_settings()
print("root                :", ROOT)
print("foundry endpoint set:", bool(settings.foundry_project_endpoint))
print("model               :", settings.foundry_model_deployment_name)
print("GITHUB_TOKEN set    :", bool(os.environ.get("GITHUB_TOKEN")), "- unauthenticated is ~10 req/min")


root                : c:\LearnForgeAI
foundry endpoint set: True
model               : gpt-5-mini
GITHUB_TOKEN set    : False - unauthenticated is ~10 req/min


## 1. Two capabilities

`web_search(query, domains=None)` and `fetch_page(url)`. Microsoft Learn and GitHub are not
services the agent calls; they are **domains it may ask for**, and the router picks the adapter.

⚠️ **`domains=` cannot be a query operator, so it is implemented as routing.** Measured on the
hosted tool: its schema is `{'type': 'web_search'}` — no parameters — and `site:` is advisory at
best. A `site:learn.microsoft.com` query leaked a `github.com` citation, and `site:wikipedia.org`
returned **zero** citations. Passing a domain filter into the query is a promise the transport
cannot keep.

So: `learn.microsoft.com` routes to the Learn API, `github.com` to the GitHub API — those APIs
*are* the domain restriction — and any other domain goes to generic search and is then filtered
client-side. If the filter empties the list we report zero hits rather than quietly handing back
unfiltered noise.


In [3]:
import httpx
from pydantic import BaseModel, Field


class SearchHit(BaseModel):
    title: str
    url: str
    snippet: str = ""
    provider: str = ""


LEARN_URL = "https://learn.microsoft.com/api/search"


async def search_learn(query: str, limit: int = 8) -> list[SearchHit]:
    params = {"search": query, "locale": "en-us", "$top": limit}
    async with httpx.AsyncClient(timeout=20) as client:
        response = await client.get(LEARN_URL, params=params)
        response.raise_for_status()
        results = response.json().get("results", [])
    return [
        SearchHit(
            title=hit.get("title") or "",
            url=hit.get("url") or "",
            snippet=hit.get("description") or "",
            provider="learn",
        )
        for hit in results
    ]


for hit in await search_learn("Microsoft Agent Framework"):
    print(f"{hit.title[:60]:<62} {hit.url}")


Step 1: Your First Agent                                       https://learn.microsoft.com/en-us/agent-framework/get-started/your-first-agent
Agent Framework documentation                                  https://learn.microsoft.com/en-us/agent-framework/
Hosted agents in Foundry Agent Service - Microsoft Foundry     https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents
What is Microsoft Foundry Agent Service? - Microsoft Foundry   https://learn.microsoft.com/en-us/azure/foundry/agents/overview
Agent Builder in Microsoft 365 Copilot                         https://learn.microsoft.com/en-us/microsoft-365/copilot/extensibility/agent-builder
Agent capabilities                                             https://learn.microsoft.com/en-us/agent-framework/agents/
Cloud Adoption Framework for Microsoft - Cloud Adoption Fram   https://learn.microsoft.com/en-us/azure/cloud-adoption-framework/
Microsoft 365 developer documentation - Microsoft 365 Develo   https://learn.mi

In [4]:
GITHUB_URL = "https://api.github.com/search/repositories"


def github_headers() -> dict[str, str]:
    headers = {"Accept": "application/vnd.github+json"}
    token = os.environ.get("GITHUB_TOKEN")
    if token:
        headers["Authorization"] = f"Bearer {token}"
    return headers


async def github_pass(client: httpx.AsyncClient, query: str, limit: int) -> list[SearchHit]:
    response = await client.get(
        GITHUB_URL, params={"q": query, "per_page": limit}, headers=github_headers()
    )
    if response.status_code >= 400:
        print(f"  github {response.status_code}: {str(response.json().get('message', ''))[:70]}")
        return []
    return [
        SearchHit(
            title=item["full_name"],
            url=item["html_url"],
            snippet=f"{(item.get('description') or '')[:150]} [{item.get('stargazers_count', 0)} stars]",
            provider="github",
        )
        for item in response.json().get("items", [])
    ]


async def search_github(query: str, limit: int = 4) -> list[SearchHit]:
    """Scoped pass first. GitHub ranks on README text, so sample repos bury the canonical one:
    best-match and sort=stars both OMIT microsoft/agent-framework entirely, while
    `in:name,description` ranks it #1. Concatenation order decides who wins de-duplication."""
    async with httpx.AsyncClient(timeout=20) as client:
        named = await github_pass(client, f"{query} in:name,description", limit)
        broad = await github_pass(client, query, limit)
    return named + broad


for hit in await search_github("Microsoft Agent Framework"):
    print(f"{hit.title[:34]:<36} {hit.snippet[-12:]:>12}  {hit.url}")


microsoft/autogen                    60359 stars]  https://github.com/microsoft/autogen
microsoft/agent-framework            12733 stars]  https://github.com/microsoft/agent-framework
rwjdk/MicrosoftAgentFrameworkSampl    [310 stars]  https://github.com/rwjdk/MicrosoftAgentFrameworkSamples
microsoft/PromptWizard               [4003 stars]  https://github.com/microsoft/PromptWizard
rwjdk/MicrosoftAgentFrameworkSampl    [310 stars]  https://github.com/rwjdk/MicrosoftAgentFrameworkSamples
Azure-Samples/interview-coach-agen    [154 stars]  https://github.com/Azure-Samples/interview-coach-agent-framework
microsoft/azure-trust-agents         k [83 stars]  https://github.com/microsoft/azure-trust-agents
webmaxru/awesome-microsoft-agent-f    [159 stars]  https://github.com/webmaxru/awesome-microsoft-agent-framework


In [5]:
from backend.services.foundry import get_chat_client

chat_client = get_chat_client()
print("client             :", type(chat_client).__name__)
print("get_web_search_tool:", hasattr(chat_client, "get_web_search_tool"))

tool = chat_client.get_web_search_tool()
print("tool               :", type(tool).__name__, "->", tool)


client             : FoundryChatClient
get_web_search_tool: True
tool               : WebSearchTool -> {'type': 'web_search'}


In [6]:
search_agent = chat_client.as_agent(
    name="subject-search-probe",
    instructions="Search the web for the name given. Cite what you find. Keep prose short.",
    tools=[tool],
)
probe = await search_agent.run(
    "What is Microsoft Agent Framework? Cite the official documentation and the project's repository."
)

print("response :", type(probe).__name__)
print("messages :", len(probe.messages))
for message in probe.messages:
    for content in message.contents:
        annotations = getattr(content, "annotations", None) or []
        print(f"  {type(content).__name__:<22} annotations={len(annotations)}")
        for annotation in annotations:
            print(f"      {type(annotation).__name__}: {getattr(annotation, 'url', None)}")


response : AgentResponse
messages : 1
  Content                annotations=0
  Content                annotations=0
  Content                annotations=0
  Content                annotations=0
  Content                annotations=4
      dict: None
      dict: None
      dict: None
      dict: None


In [29]:
def annotation_field(annotation, field: str) -> str | None:
    """Annotations arrive as plain dicts. Reading them as attributes returns None for every
    citation and yields zero hits with no error - the same silent empty as calling the Learn
    MCP tool with the wrong parameter name. Handle both shapes so a version bump cannot go quiet.
    """
    if isinstance(annotation, dict):
        return annotation.get(field)
    return getattr(annotation, field, None)


def url_citations(response) -> list[SearchHit]:
    """Keep the citation objects and discard the prose: the URLs come from the tool rather than
    being retyped by the model, so a mistyped path is unrepresentable."""
    found: list[SearchHit] = []
    for message in getattr(response, "messages", None) or []:
        for content in getattr(message, "contents", None) or []:
            for annotation in getattr(content, "annotations", None) or []:
                url = annotation_field(annotation, "url")
                if url:
                    found.append(
                        SearchHit(
                            title=annotation_field(annotation, "title") or url,
                            url=url,
                            provider="generic",
                        )
                    )
    return found


async def search_generic(query: str, limit: int = 8) -> list[SearchHit]:
    """The discovery provider. It reaches domains our two APIs cannot - rust-lang.org,
    react.dev, spark.apache.org - which is exactly what a first search needs to find."""
    response = await search_agent.run(
        f"What is {query}? Cite the official documentation and the project's own site or repository."
    )
    seen: set[str] = set()
    hits: list[SearchHit] = []
    for hit in url_citations(response):
        if hit.url not in seen:
            seen.add(hit.url)
            hits.append(hit)
    return hits[:limit]


assert url_citations(probe), "citations must survive extraction - if this is empty the shape moved"
for hit in url_citations(probe):
    print(f"{hit.title[:52]:<54} {hit.url}")


Microsoft Agent Framework Overview | Microsoft Learn   https://learn.microsoft.com/en-us/agent-framework/overview/
Microsoft Agent Framework Overview | Microsoft Learn   https://learn.microsoft.com/en-us/agent-framework/overview/
Agent Framework documentation | Microsoft Learn        https://learn.microsoft.com/en-us/agent-framework/
GitHub - microsoft/agent-framework: A framework for    https://github.com/microsoft/agent-framework


In [30]:
from urllib.parse import urlparse

MAX_SEARCHES = 3
MAX_FETCHES = 5

# The only two domains that have an API precise enough to BE the domain restriction.
DOMAIN_ADAPTERS = {
    "learn.microsoft.com": search_learn,
    "github.com": search_github,
}


def host_of(url: str) -> str:
    return (urlparse(url).hostname or "").lower().removeprefix("www.")


def on_domains(hit: SearchHit, domains: list[str]) -> bool:
    host = host_of(hit.url)
    return any(host == d or host.endswith(f".{d}") for d in (x.lower() for x in domains))


def dedupe(hits: list[SearchHit]) -> list[SearchHit]:
    seen: set[str] = set()
    out: list[SearchHit] = []
    for hit in hits:
        key = hit.url.rstrip("/")
        if key and key not in seen:
            seen.add(key)
            out.append(hit)
    return out


class Trace(BaseModel):
    """Every search and fetch the run actually performed. The model picks the strategy; this
    records what code did about it, so a refusal can be told apart from a run that never looked."""

    searches: list[str] = Field(default_factory=list)
    fetches: list[str] = Field(default_factory=list)
    notes: list[str] = Field(default_factory=list)

    @property
    def searches_left(self) -> int:
        return MAX_SEARCHES - len(self.searches)

    @property
    def fetches_left(self) -> int:
        return MAX_FETCHES - len(self.fetches)


async def web_search(
    query: str, domains: list[str] | None = None, trace: Trace | None = None
) -> list[SearchHit]:
    """One capability. `domains=None` is generic discovery; a domain list routes to an adapter,
    or falls back to generic search filtered client-side."""
    label = f"{query!r} domains={domains or 'any'}"
    if trace is not None:
        if trace.searches_left <= 0:
            trace.notes.append(f"search budget spent, refused {label}")
            return []
        trace.searches.append(label)

    if not domains:
        return await search_generic(query)

    adapters = [DOMAIN_ADAPTERS[d] for d in domains if d in DOMAIN_ADAPTERS]
    if adapters:
        gathered = await asyncio.gather(*(call(query) for call in adapters), return_exceptions=True)
        hits = [h for result in gathered if not isinstance(result, BaseException) for h in result]
        return dedupe(hits)

    # No adapter for these domains, so search generally and filter. An empty result is reported
    # as empty; handing back unfiltered noise would silently answer a question nobody asked.
    hits = [hit for hit in await search_generic(query) if on_domains(hit, domains)]
    if not hits and trace is not None:
        trace.notes.append(f"no results survived the domain filter for {domains}")
    return dedupe(hits)


print(f"budget: {MAX_SEARCHES} searches, {MAX_FETCHES} fetches")
print("adapters:", list(DOMAIN_ADAPTERS))


budget: 3 searches, 5 fetches
adapters: ['learn.microsoft.com', 'github.com']


## 2. Fetching

Finding needs providers; **fetching does not** — one path serves every URL.

Two things here are load-bearing rather than defensive:

- `follow_redirects=True` is required, or every Learn link dies on its 302 to `/en-us/...`.
- `is_fetchable` is a real SSRF boundary, not theatre. We are about to fetch pages a search engine
  chose, and `DefaultAzureCredential` itself probes `169.254.169.254` — the exact address this
  blocks — so we know that target is reachable from inside.


In [31]:
from ipaddress import ip_address
from urllib.parse import urlparse

MAX_HTML_CHARS = 400_000
MAX_TEXT_CHARS = 40_000
MIN_WORDS = 50
BLOCKED_SUFFIXES = ("localhost", ".internal", ".local")

# Measured: Wikimedia 403s a vague contact string on both the API and REST endpoints, and returns
# 200 as soon as the User-Agent carries a contact URL. Set once, here, for every fetch - the last
# run fixed this in the search provider and left the fetcher behind, so every Wikipedia read 403'd.
USER_AGENT = "LearnForgeAI-probe/0.1 (https://github.com/learnforge; research probe)"


def is_fetchable(url: str) -> bool:
    parsed = urlparse(url)
    if parsed.scheme != "https" or not parsed.hostname:
        return False
    host = parsed.hostname.lower()
    if host.endswith(BLOCKED_SUFFIXES):
        return False
    try:
        return ip_address(host).is_global
    except ValueError:
        return True


# Blocks go first WITH their contents; stripping tags alone leaves nav and script text behind.
STRIP_BLOCKS = re.compile(r"<(script|style|nav|footer|header|svg)\b.*?</\1>", re.S | re.I)
TAGS = re.compile(r"<[^>]+>")
ENTITIES = re.compile(r"&[a-zA-Z#0-9]+;")
WHITESPACE = re.compile(r"\s+")


def extract_text(html: str) -> str:
    text = STRIP_BLOCKS.sub(" ", html)
    text = TAGS.sub(" ", text)
    text = ENTITIES.sub(" ", text)
    return WHITESPACE.sub(" ", text).strip()[:MAX_TEXT_CHARS]


class SourceDocument(BaseModel):
    title: str
    url: str
    text: str

    @property
    def words(self) -> int:
        return len(self.text.split())


async def fetch_one(client: httpx.AsyncClient, hit: SearchHit) -> SourceDocument | None:
    try:
        response = await client.get(hit.url)
    except Exception as error:
        print(f"  fetch failed {type(error).__name__:<18} {hit.url}")
        return None
    if response.status_code >= 400:
        print(f"  fetch {response.status_code}     {hit.url}")
        return None
    return SourceDocument(
        title=hit.title, url=hit.url, text=extract_text(response.text[:MAX_HTML_CHARS])
    )


async def fetch_documents(hits: list[SearchHit], limit: int = MAX_FETCHES) -> list[SourceDocument]:
    usable = [hit for hit in hits if is_fetchable(hit.url)][:limit]
    async with httpx.AsyncClient(
        timeout=25, follow_redirects=True, headers={"User-Agent": USER_AGENT}
    ) as client:
        fetched = await asyncio.gather(*(fetch_one(client, hit) for hit in usable))
    return [doc for doc in fetched if doc is not None and doc.words > MIN_WORDS]


assert not is_fetchable("https://169.254.169.254/metadata/identity")
assert not is_fetchable("http://learn.microsoft.com/agent-framework/")
assert is_fetchable("https://learn.microsoft.com/agent-framework/")
print("SSRF gate ok; user-agent carries a contact URL")


SSRF gate ok; user-agent carries a contact URL


## 3. The schema

Built as specified, including `confidence`, `difficulty` and `estimated_hours`. Three of those
have a prior against them, so they are **instrumented rather than argued about** — §7 measures
whether they separate signal from noise, and the data decides.

| Field | Prior evidence | How it is settled here |
|---|---|---|
| `confidence: float` | Its failure mode is *accept the wrong material*: a wrong identity at 0.95 is worse than no field. The same model scored 5/5 on a `bool` escape hatch — uncertainty worked as a branch, not as a dial. | §7 compares it on correct vs incorrect identifications. If both read ~0.95 no threshold can gate. |
| `difficulty` | Returned `intermediate` for 4 of 6 topics, including an invented one. | §7 checks it across 3 runs on identical evidence. |
| `estimated_hours` | The model choosing our cost — 150 hours became 20 chapters, ~10 months at 30 min/day. | §7 reports the spread across runs. |

Two constraints carried in from earlier failures, and these are **not** negotiable by measurement
because both have already been paid for:

- **`evidence` cites a document number, never a URL.** We number the documents, so a mistyped URL
  is unrepresentable.
- **`canonical_name` may only be a name the documents themselves use.** Asked for aliases from
  memory, this model put *"Microsoft Agent"* in MAF's alias list — which would have made us accept
  Clippy material as on-topic. Aliases may help recall; they must never grant acceptance.
  `Azure AI Search` is in the subject list precisely to exercise a genuinely renamed product.

`TechnicalSubjectType` keeps `OTHER`: a closed set with no escape hatch is just a demand for the
nearest neighbour.


In [32]:
from enum import StrEnum


class IdentityStatus(StrEnum):
    CONFIRMED = "confirmed"
    AMBIGUOUS = "ambiguous"
    UNRECOGNISED = "unrecognised"


class TechnicalSubjectType(StrEnum):
    PROGRAMMING_LANGUAGE = "programming_language"
    SOFTWARE_FRAMEWORK = "software_framework"
    SOFTWARE_LIBRARY = "software_library"
    PLATFORM = "platform"
    SERVICE = "service"
    TOOL = "tool"
    PROTOCOL_OR_SPECIFICATION = "protocol_or_specification"
    CONCEPT_OR_PRACTICE = "concept_or_practice"
    PRODUCT_FEATURE = "product_feature"
    OTHER = "other"


class ExperienceLevel(StrEnum):
    BEGINNER = "beginner"
    INTERMEDIATE = "intermediate"
    ADVANCED = "advanced"


class SubjectEvidence(BaseModel):
    document_index: int = Field(
        description="The number of the document that supports this, as printed. Never a URL."
    )
    supporting_claim: str = Field(
        description="What that document actually says which establishes what this subject is."
    )


class SubjectAnalysis(BaseModel):
    identity_status: IdentityStatus = Field(
        description=(
            "confirmed when the documents describe the subject that was asked for; ambiguous when "
            "they describe several unrelated technical subjects sharing that name; unrecognised "
            "when none of them describes it."
        )
    )
    confidence: float = Field(
        ge=0.0, le=1.0, description="How strongly this evidence establishes the identity."
    )
    canonical_name: str | None = Field(
        default=None,
        description=(
            "The name the documents themselves use for this subject, including a current name "
            "that has replaced an older one. Null unless a document states it."
        ),
    )
    subject_type: TechnicalSubjectType = Field(
        description="What kind of technical thing it is, per the documents."
    )
    domain: str = Field(
        default="", description="Broad technical field, e.g. 'Artificial Intelligence'."
    )
    subdomain: str | None = Field(default=None, description="Narrower area, e.g. 'Agentic AI'.")
    description: str = Field(
        default="", description="One or two sentences on what it is, drawn from the documents."
    )
    scope: list[str] = Field(
        default_factory=list,
        description=(
            "The main areas this subject covers, named as the documents name them. Short noun "
            "phrases such as 'workflows' or 'middleware', not sentences."
        ),
    )
    prerequisites: list[str] = Field(
        default_factory=list,
        description=(
            "What genuinely blocks starting. Never installation or setup steps, general computer "
            "literacy, or ordinary tools such as a text editor, a browser or a git host."
        ),
    )
    difficulty: ExperienceLevel = Field(
        description="How hard the subject is in itself, independent of any particular learner."
    )
    estimated_hours: int = Field(
        ge=1, le=500, description="Hours to reach working competence in the requested subject."
    )
    candidates: list[str] = Field(
        default_factory=list,
        description=(
            "Only when ambiguous: the distinct technical subjects the documents describe under "
            "this name. Do not choose between them."
        ),
    )
    evidence: list[SubjectEvidence] = Field(
        default_factory=list, description="Which documents established the identity, and how."
    )


print(len(TechnicalSubjectType), "subject types,", len(IdentityStatus), "identity states")


10 subject types, 3 identity states


In [33]:
class TargetedSearch(BaseModel):
    query: str = Field(description="The search to run.")
    domains: list[str] = Field(
        default_factory=list,
        description=(
            "Restrict to these hostnames, e.g. ['learn.microsoft.com'] or ['rust-lang.org']. "
            "Leave empty for a general search."
        ),
    )


class SearchPlan(BaseModel):
    """What to do next, as data. Code executes it, so the plan and the actions cannot diverge."""

    assessment: str = Field(
        description="One sentence on what these results appear to show about the subject."
    )
    looks_ambiguous: bool = Field(
        description=(
            "True when the results describe several unrelated technical subjects sharing the name "
            "and nothing selects between them."
        )
    )
    fetch: list[int] = Field(
        default_factory=list,
        description=(
            "Numbers of the results worth reading in full, strongest evidence first. Prefer "
            "first-party documentation and the project's own repository. Two or three is usually "
            "enough; never pick a result just to fill the list."
        ),
    )
    targeted_searches: list[TargetedSearch] = Field(
        default_factory=list,
        description=(
            "Only when these results leave the identity unsettled. Leave empty when what is here "
            "already answers it - a search you do not need is latency the learner pays for."
        ),
    )


PLANNER_PROMPT = """You are deciding what to read in order to establish what a technical subject is.

You are given the results of one general web search. Your job is NOT to say what the subject is
yet - it is to choose the smallest set of sources that would settle it.

Prefer, in order:
1. First-party documentation for the named thing.
2. The project's own repository or website.
3. A specification or standards document, when the subject is a protocol or a practice rather
   than a product.
4. A reputable secondary source, only when nothing above is present.

Rules:

- Select results by NUMBER. You are choosing from a list you were given, not recalling URLs.
- Stop early. If two results already establish the identity, ask for those two and no more.
- Request a targeted search only when the results genuinely leave the identity unsettled. Say
  which domain to search when a first-party source is likely but missing from these results.
- Set looks_ambiguous when several unrelated technical subjects share this name here. Do not try
  to break the tie yourself - that is the learner's choice to make, not yours.
- Results that describe a DIFFERENT product with a similar name are evidence of absence. A search
  engine returns its best guess rather than nothing, so a full list is not proof the subject is real.
"""


@lru_cache
def get_planner_agent():
    return get_chat_client().as_agent(
        name="subject-planner-probe",
        instructions=PLANNER_PROMPT,
        default_options={"response_format": SearchPlan},
    )


def number_hits(hits: list[SearchHit]) -> str:
    return "\n".join(
        f"[{n}] {hit.title}\n    {hit.url}\n    {hit.snippet[:180]}"
        for n, hit in enumerate(hits, start=1)
    )


async def plan_next(subject: str, hits: list[SearchHit]) -> SearchPlan:
    prompt = (
        f"The learner asked to learn: {subject}\n\n"
        f"A general web search returned {len(hits)} results:\n\n{number_hits(hits)}"
    )
    return (await get_planner_agent().run(prompt)).value


print("planner ready:", SearchPlan.model_fields.keys())


planner ready: dict_keys(['assessment', 'looks_ambiguous', 'fetch', 'targeted_searches'])


## 4. The analyser and the loop

Two model calls per subject on the normal path — plan, then analyse — plus one more only when the
plan asks for a targeted search. `1 search + 2 fetches + 1 reasoning call` is the expected shape.

The analyser is given the fetched text and asked what **these documents** say. It is never asked
what the subject *is*. That distinction is why node 1 works, and why three separate attempts to
ask this model about a subject it had not heard of all failed.


In [34]:
CHARS_PER_DOCUMENT = 6_000

ANALYSER_PROMPT = """You identify what a technical subject is, using only documents that have
already been retrieved and read for you.

You cannot search. Judge only what is in front of you, and do not use what you remember about the
name - a name you half-recognise is exactly where this goes wrong.

Set identity_status to:

- confirmed    when the documents describe the subject that was asked for.
- ambiguous    when they describe several unrelated technical subjects sharing the name. List them
               in candidates and do not choose between them.
- unrecognised when none of the documents describes the requested name. A search engine returns
               its best guess rather than nothing, so documents about a DIFFERENT product with a
               similar name are evidence of absence, not evidence of presence.

You are not answering a question about the world; you are reporting what this evidence supports.
Saying the evidence does not establish it is the most valuable answer you can give.

canonical_name may only be a name these documents actually use. If a document says the product was
renamed, the current name is the canonical one. Never supply a name from memory.

Every entry in evidence cites a document by its printed NUMBER and quotes or closely paraphrases
what that document says. Never cite a document for a claim it does not make.
"""


@lru_cache
def get_analyser_agent():
    return get_chat_client().as_agent(
        name="subject-analyser-probe",
        instructions=ANALYSER_PROMPT,
        default_options={"response_format": SubjectAnalysis},
    )


def number_documents(docs: list[SourceDocument]) -> str:
    return "\n\n".join(
        f"[{n}] {doc.title}\n{doc.url}\n{doc.text[:CHARS_PER_DOCUMENT]}"
        for n, doc in enumerate(docs, start=1)
    )


async def analyse_documents(subject: str, docs: list[SourceDocument]) -> SubjectAnalysis:
    prompt = (
        f"The learner asked to learn: {subject}\n\n"
        f"{len(docs)} documents were retrieved and read for that name:\n\n{number_documents(docs)}"
    )
    return (await get_analyser_agent().run(prompt)).value


print("analyser ready:", len(SubjectAnalysis.model_fields), "fields")


analyser ready: 13 fields


In [41]:
class Investigation(BaseModel):
    subject: str
    analysis: SubjectAnalysis | None = None
    trace: Trace = Field(default_factory=Trace)
    documents: list[SourceDocument] = Field(default_factory=list)
    plans: list[SearchPlan] = Field(default_factory=list)
    failure: str | None = None


def pick(hits: list[SearchHit], numbers: list[int], budget: int) -> list[SearchHit]:
    """Indexes, not URLs. Out-of-range numbers are dropped rather than wrapped - a silent
    modulo would hand back a source the model never chose."""
    chosen: list[SearchHit] = []
    for number in numbers:
        if 1 <= number <= len(hits) and hits[number - 1] not in chosen:
            chosen.append(hits[number - 1])
        if len(chosen) >= budget:
            break
    return chosen


def as_documents(hits: list[SearchHit]) -> list[SourceDocument]:
    """Search results demoted to thin documents, used only when no page could be read."""
    return [
        SourceDocument(title=hit.title, url=hit.url, text=f"{hit.title}. {hit.snippet}".strip())
        for hit in hits
    ]


async def investigate(subject: str) -> Investigation:
    run = Investigation(subject=subject)
    trace = run.trace

    hits = await web_search(subject, trace=trace)
    if not hits:
        run.failure = "the discovery search returned nothing"
        return run

    plan = await plan_next(subject, hits)
    run.plans.append(plan)

    # A targeted search only happens because the plan asked for one, and only within budget.
    for targeted in plan.targeted_searches:
        if trace.searches_left <= 0:
            break
        extra = await web_search(targeted.query, targeted.domains or None, trace=trace)
        hits = dedupe(hits + extra)
    if run.plans[0].targeted_searches and hits:
        plan = await plan_next(subject, hits)
        run.plans.append(plan)

    selected = pick(hits, plan.fetch, trace.fetches_left)
    if not selected:
        trace.notes.append("the plan selected no source; falling back to the top readable hits")
        selected = [hit for hit in hits if is_fetchable(hit.url)][:2]

    run.documents = await fetch_documents(selected, limit=trace.fetches_left)
    trace.fetches.extend(doc.url for doc in run.documents)

    # ⚠️ Measured: for an invented subject the search returns junk (an npmjs.com root page) that
    # then 403s, so "nothing exists" and "retrieval broke" arrive as the same empty list. Returning
    # no verdict is the worst of the three options - the caller cannot tell a refusal from a crash.
    # Judge the titles and snippets instead, and record that the verdict rests on thin evidence.
    evidence = run.documents
    if not evidence:
        trace.notes.append("no page could be read; judging search titles and snippets only")
        evidence = as_documents(hits[:MAX_FETCHES])
    if not evidence:
        run.failure = "nothing was found and nothing could be read"
        return run

    run.analysis = await analyse_documents(subject, evidence)
    return run


def report(run: Investigation) -> None:
    analysis = run.analysis
    print(f"=== {run.subject}")
    for query in run.trace.searches:
        print(f"  search  {query}")
    for note in run.trace.notes:
        print(f"  note    {note}")
    for plan in run.plans:
        print(f"  plan    ambiguous={plan.looks_ambiguous} fetch={plan.fetch} :: {plan.assessment}")
    for url in run.trace.fetches:
        print(f"  read    {url}")
    if analysis is None:
        print(f"  FAILED  {run.failure}")
        return
    print(f"  status  {analysis.identity_status}  confidence={analysis.confidence}")
    print(f"  name    {analysis.canonical_name}   type={analysis.subject_type}")
    print(f"  domain  {analysis.domain} / {analysis.subdomain}")
    print(f"  sizing  {analysis.difficulty}, {analysis.estimated_hours}h")
    print(f"  scope   {', '.join(analysis.scope) or '-'}")
    print(f"  prereq  {', '.join(analysis.prerequisites) or '-'}")
    print(f"  cands   {', '.join(analysis.candidates) or '-'}")
    for item in analysis.evidence:
        ok = 1 <= item.document_index <= len(run.documents)
        cited = run.documents[item.document_index - 1].url if ok else "(snippet-only evidence)"
        print(f"    [{item.document_index}] {cited}")


print("loop ready")


loop ready


In [ ]:
# Change the name to probe a single subject end to end without re-running the matrix.
report(await investigate("Microsoft Agent Framework"))


=== Blorptagon SDK
  search  'Blorptagon SDK' domains=any
  search  'Blorptagon SDK' domains=any
  search  'Blorptagon SDK' domains=['github.com', 'npmjs.com', 'pypi.org', 'blorptagon.com']
  note    the plan selected no source; falling back to the top readable hits
  plan    ambiguous=False fetch=[1] :: The single result is a YouTube short about a 'Blorptagon Dilemma' — no sign of an SDK or first‑party docs, so the identity of 'Blorptagon SDK' is unclear.
  plan    ambiguous=False fetch=[] :: The two results do not show an official SDK — one is an unrelated YouTube short and the other is the GitHub homepage — so the identity of “Blorptagon SDK” is unresolved.
  read    https://github.com/
  status  unrecognised  confidence=0.7
  name    None   type=other
  domain  Unknown / None
  sizing  beginner, 1h
  scope   -
  prereq  -
  cands   -
    [1] https://github.com/


## 5. The subject set

Technical only. `Statistics` and `Guitar` are gone — they were the two failures last time, and
dropping them is the scope decision, not a fix. Replacing them are subjects that stress the parts
of this design that are actually new:

| Subject | What it tests |
|---|---|
| `Microsoft Agent Framework` | the original failure — Bot Framework, then Clippy |
| `Azure AI Search` | a genuinely renamed product; `canonical_name` must come from a document, not memory |
| `React`, `Rust`, `Apache Spark` | first-party site that is neither Learn nor GitHub — the routing case |
| `OAuth 2.0` | a specification with no product owner |
| `Kubernetes operators`, `Excel pivot tables` | descriptive names that no page quotes verbatim |
| `Agent Framework`, `Agent` | must come back ambiguous, not resolved |
| `Blorptagon SDK` | invented; search still offers `plotagon.com` and `broctagon.com` |

Each is judged **3 times**. The acceptance criterion is stability, not a single pass — the
tool-loop design passed Rust twice and failed it on the third run.

⚠️ Unlike the last build, the search is **inside** the loop, so re-running re-searches. That is
deliberate here: the routing decision is part of what we are measuring, and caching it would hide
the variance we care about most.


In [37]:
SUBJECTS: list[tuple[str, str, str]] = [
    ("Microsoft Agent Framework", "confirmed", "the original failure"),
    ("Azure AI Search", "confirmed", "renamed from Azure Cognitive Search"),
    ("Rust", "confirmed", "first-party site is neither Learn nor GitHub"),
    ("React", "confirmed", "react.dev - routing to an arbitrary domain"),
    ("Apache Spark", "confirmed", "spark.apache.org"),
    ("Kubernetes", "confirmed", "well covered; should be cheap"),
    ("Python", "confirmed", "also a snake"),
    ("OAuth 2.0", "confirmed", "a specification, no product owner"),
    ("Kubernetes operators", "confirmed", "descriptive name, never quoted verbatim"),
    ("Excel pivot tables", "confirmed", "descriptive name, product feature"),
    ("Agent Framework", "ambiguous", "MAF vs OpenAI Agents SDK vs others"),
    ("Agent", "ambiguous", "too generic to resolve"),
    ("Blorptagon SDK", "unrecognised", "invented"),
]

RUNS = 3
EXPECTED = {name: expected for name, expected, _ in SUBJECTS}

for name, expected, why in SUBJECTS:
    print(f"{name:<28} {expected:<14} {why}")


Microsoft Agent Framework    confirmed      the original failure
Azure AI Search              confirmed      renamed from Azure Cognitive Search
Rust                         confirmed      first-party site is neither Learn nor GitHub
React                        confirmed      react.dev - routing to an arbitrary domain
Apache Spark                 confirmed      spark.apache.org
Kubernetes                   confirmed      well covered; should be cheap
Python                       confirmed      also a snake
OAuth 2.0                    confirmed      a specification, no product owner
Kubernetes operators         confirmed      descriptive name, never quoted verbatim
Excel pivot tables           confirmed      descriptive name, product feature
Agent Framework              ambiguous      MAF vs OpenAI Agents SDK vs others
Agent                        ambiguous      too generic to resolve
Blorptagon SDK               unrecognised   invented


In [38]:
import random

# Resumable on purpose. The last matrix died on subject 2 of 15 with `getaddrinfo failed` and took
# every completed subject with it, because the results dict was rebuilt at the top of the cell.
RESULTS: dict[str, list[Investigation]] = globals().get("RESULTS", {})

MAX_ATTEMPTS = 3
TRANSIENT = ("getaddrinfo", "connection", "timeout", "temporarily", "429", "503")


def is_transient(error: Exception) -> bool:
    text = f"{type(error).__name__} {error}".lower()
    return any(mark in text for mark in TRANSIENT)


async def investigate_with_retry(subject: str) -> Investigation:
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            return await investigate(subject)
        except Exception as error:
            if attempt == MAX_ATTEMPTS or not is_transient(error):
                raise
            delay = 2**attempt + random.random()
            print(f"    {type(error).__name__} on attempt {attempt}, retrying in {delay:.1f}s")
            await asyncio.sleep(delay)
    raise RuntimeError("unreachable")


# The three runs of one subject go in parallel; subjects stay serial so a rate limit cannot
# cascade across the whole matrix.
for name, _, _ in SUBJECTS:
    if len(RESULTS.get(name, [])) == RUNS:
        continue
    gathered = await asyncio.gather(
        *(investigate_with_retry(name) for _ in range(RUNS)), return_exceptions=True
    )
    runs = [r for r in gathered if isinstance(r, Investigation)]
    if not runs:
        first = gathered[0]
        print(f"{name:<28} ABANDONED {type(first).__name__}: {str(first)[:60]}")
        continue
    RESULTS[name] = runs
    verdicts = [str(r.analysis.identity_status) if r.analysis else "FAILED" for r in runs]
    searches = [len(r.trace.searches) for r in runs]
    fetches = [len(r.trace.fetches) for r in runs]
    print(f"{name:<26} {str(verdicts):<46} searches={searches} fetches={fetches}")

print(f"\n{len(RESULTS)}/{len(SUBJECTS)} subjects complete")


Microsoft Agent Framework  ['confirmed', 'confirmed', 'confirmed']        searches=[1, 1, 1] fetches=[2, 2, 2]
Azure AI Search            ['confirmed', 'confirmed', 'confirmed']        searches=[1, 1, 1] fetches=[2, 2, 2]
Rust                       ['confirmed', 'confirmed', 'confirmed']        searches=[1, 1, 1] fetches=[2, 2, 2]
React                      ['confirmed', 'confirmed', 'confirmed']        searches=[1, 1, 1] fetches=[2, 2, 2]
Apache Spark               ['confirmed', 'confirmed', 'confirmed']        searches=[1, 1, 1] fetches=[2, 2, 2]
Kubernetes                 ['confirmed', 'confirmed', 'confirmed']        searches=[1, 1, 1] fetches=[2, 2, 1]
Python                     ['confirmed', 'confirmed', 'confirmed']        searches=[1, 1, 1] fetches=[2, 2, 2]
OAuth 2.0                  ['confirmed', 'confirmed', 'confirmed']        searches=[1, 1, 1] fetches=[2, 2, 1]
Kubernetes operators       ['confirmed', 'confirmed', 'confirmed']        searches=[1, 1, 1] fetches=[2, 2, 1]
E

In [39]:
print(f"{'subject':<28} {'expected':<13} {'got':<13} {'stable':<7} {'search':<7} {'fetch':<6} conf")
print("-" * 88)

failures: list[str] = []
for name, expected, _ in SUBJECTS:
    runs = RESULTS.get(name, [])
    if not runs:
        print(f"{name:<28} {expected:<13} {'NOT RUN':<13}")
        failures.append(f"{name}: not run")
        continue
    statuses = [str(r.analysis.identity_status) if r.analysis else "FAILED" for r in runs]
    common = Counter(statuses).most_common(1)[0][0]
    stable = len(set(statuses)) == 1
    searches = sum(len(r.trace.searches) for r in runs) / len(runs)
    fetches = sum(len(r.trace.fetches) for r in runs) / len(runs)
    confs = [r.analysis.confidence for r in runs if r.analysis]
    conf = f"{sum(confs) / len(confs):.2f}" if confs else "-"
    print(
        f"{name:<28} {expected:<13} {common:<13} {('yes' if stable else 'NO'):<7} "
        f"{searches:<7.1f} {fetches:<6.1f} {conf}"
    )
    if not stable:
        failures.append(f"{name}: unstable {statuses}")
    elif common != expected:
        failures.append(f"{name}: expected {expected}, got {common}")

judged = [r for runs in RESULTS.values() for r in runs if r.analysis]
if judged:
    print(
        f"\ncost per run: {sum(len(r.trace.searches) for r in judged) / len(judged):.2f} searches, "
        f"{sum(len(r.trace.fetches) for r in judged) / len(judged):.2f} fetches"
    )
print()
if failures:
    print("NOT READY TO IMPLEMENT:")
    for failure in failures:
        print(" -", failure)
else:
    print(f"All {len(SUBJECTS)} subjects stable and correct across {RUNS} runs.")


subject                      expected      got           stable  search  fetch  conf
----------------------------------------------------------------------------------------
Microsoft Agent Framework    confirmed     confirmed     yes     1.0     2.0    0.90
Azure AI Search              confirmed     confirmed     yes     1.0     2.0    0.92
Rust                         confirmed     confirmed     yes     1.0     2.0    0.92
React                        confirmed     confirmed     yes     1.0     2.0    0.92
Apache Spark                 confirmed     confirmed     yes     1.0     2.0    0.92
Kubernetes                   confirmed     confirmed     yes     1.0     1.7    0.92
Python                       confirmed     confirmed     yes     1.0     2.0    0.93
OAuth 2.0                    confirmed     confirmed     yes     1.0     1.7    0.95
Kubernetes operators         confirmed     confirmed     yes     1.0     1.7    0.90
Excel pivot tables           confirmed     confirmed     yes 

## 7. Do the numeric fields carry any signal?

The scoreboard above says whether the *routing* works. This section decides whether the three
model-supplied numbers deserve to exist, using the run we just paid for.

**`confidence` is settled by separation, not by average.** The question is not "is confidence
high on correct answers" — it is whether the correct and incorrect populations are far enough
apart that a threshold sits between them. A gate at `>= 0.90` is only meaningful if wrong
identifications actually land below it. If both groups read ~0.95, the field is a dial that
cannot gate, and the honest move is to delete it rather than pick a threshold that never fires.

**`difficulty` and `estimated_hours` are settled by drift.** Each subject is judged 3 times on
evidence gathered independently, so a field that changes between runs is reporting noise. Watch
`estimated_hours` in particular: it is the number that decides chapter count, and 150 once became
20 chapters — about ten months at 30 minutes a day.


In [40]:
right = [r.analysis.confidence for n, runs in RESULTS.items() for r in runs
         if r.analysis and str(r.analysis.identity_status) == EXPECTED[n]]
wrong = [r.analysis.confidence for n, runs in RESULTS.items() for r in runs
         if r.analysis and str(r.analysis.identity_status) != EXPECTED[n]]


def band(values: list[float]) -> str:
    if not values:
        return "none"
    return f"n={len(values):<3} min={min(values):.2f} mean={sum(values) / len(values):.2f} max={max(values):.2f}"


print("confidence on CORRECT identifications :", band(right))
print("confidence on WRONG   identifications :", band(wrong))
if right and wrong:
    if min(right) > max(wrong):
        print(f"\nSEPARATED. A threshold between {max(wrong):.2f} and {min(right):.2f} would gate.")
    else:
        print(
            f"\nOVERLAPPING. Wrong answers reach {max(wrong):.2f} while correct ones fall to "
            f"{min(right):.2f}, so no threshold separates them - confidence cannot gate."
        )
elif not wrong:
    print("\nNo wrong identifications in this run, so separation is UNMEASURED.")
    print("A high average proves nothing on its own: the field was never asked to be wrong.")

print("\n--- drift on identical subjects, across runs ---")
print(f"{'subject':<28} {'difficulty':<34} {'estimated_hours':<22} scope")
for name, _, _ in SUBJECTS:
    runs = [r for r in RESULTS.get(name, []) if r.analysis]
    if not runs:
        continue
    levels = {str(r.analysis.difficulty) for r in runs}
    hours = [r.analysis.estimated_hours for r in runs]
    scopes = [{s.lower() for s in r.analysis.scope} for r in runs]
    shared, union = set.intersection(*scopes), set.union(*scopes)
    spread = f"{hours} spread={max(hours) - min(hours)}"
    print(
        f"{name:<28} {str(sorted(levels)):<34} {spread:<22} "
        f"{len(shared)}/{len(union)} agreed"
    )


confidence on CORRECT identifications : n=33  min=0.80 mean=0.91 max=0.95
confidence on WRONG   identifications : n=4   min=0.90 mean=0.93 max=0.95

OVERLAPPING. Wrong answers reach 0.95 while correct ones fall to 0.80, so no threshold separates them - confidence cannot gate.

--- drift on identical subjects, across runs ---
subject                      difficulty                         estimated_hours        scope
Microsoft Agent Framework    ['advanced', 'intermediate']       [40, 120, 40] spread=80 0/25 agreed
Azure AI Search              ['intermediate']                   [40, 30, 40] spread=10 2/19 agreed
Rust                         ['intermediate']                   [200, 150, 150] spread=50 0/27 agreed
React                        ['intermediate']                   [60, 80, 40] spread=40 1/14 agreed
Apache Spark                 ['intermediate']                   [80, 120, 80] spread=40 0/25 agreed
Kubernetes                   ['advanced', 'intermediate']       [80, 80, 120] sp

## 8. The borderline measurement

Everything above measured a prototype. This section measures **the code that shipped** —
`backend.agents.subject_analysis.investigate` — because two decisions are still open and both
need data rather than argument.

**Question A — would a first-party rule refuse good subjects?**
The proposal is "`confirmed` requires at least one first-party source". It sounds right, and it
might refuse `OAuth 2.0` (a specification with no product owner) or `Kubernetes operators`
(a concept whose canonical pages are third-party repositories) — both of which identified
correctly. We could not answer this before because we did not record the input to the rule.
`SubjectEvidence.source_kind` now exists precisely so this is answerable.

**Question B — where does the technical boundary actually fall?**
Node 1 passes `Photography` through, node 2 will confirm it, and we would build an out-of-scope
course. A gate needs a boundary, and the boundary is fuzzy: `Excel pivot tables` is already in
the confirmed set, and `SEO`, `technical writing` and `UX design` are genuinely arguable. A
false reject is user-visible and infuriating, so the line gets measured before it is drawn.

One retrieval per subject answers both: the shipped run gives `source_kind`, and a second small
call asks only the technical question over the same documents.

⚠️ These cells import from `backend`, shadowing the prototype classes defined earlier in this
notebook. Nothing above is re-run, so that is safe — but do not mix the two sets of names.


In [44]:
from backend.agents.subject_analysis import investigate as ship_investigate
from backend.services.foundry import get_chat_client as ship_client
from backend.workflow.state import IdentityStatus as ShipStatus
from backend.workflow.state import SourceKind

FIRST_PARTY = {
    SourceKind.FIRST_PARTY_DOCUMENTATION,
    SourceKind.OFFICIAL_REPOSITORY,
    SourceKind.SPECIFICATION,
}


class TechnicalJudgement(BaseModel):
    is_technical: bool = Field(
        description=(
            "True when this subject is a technology, or a practice carried out with technology, "
            "that a hands-on technical course could teach."
        )
    )
    reason: str = Field(description="One sentence, citing what the documents describe.")


TECHNICAL_PROMPT = """You decide whether a subject belongs on a technical learning platform.

You are given documents that were retrieved for the subject. Judge from those, not from memory.

The platform teaches technologies and the practices built on them: languages, frameworks,
libraries, platforms, services, tools, protocols, specifications, and engineering practices.

It does not teach subjects with no technical substance, even when software is used to practise
them - a creative art, a physical skill, an academic field, or a business discipline.

The interesting cases are the mixed ones. Ask what a learner would actually be doing: if the
course would teach a tool or a technique that only exists because of the technology, it is
technical. If the tool is incidental and the substance is aesthetic, physical or commercial, it
is not.
"""


@lru_cache
def get_technical_agent():
    return ship_client().as_agent(
        name="technical-boundary-probe",
        instructions=TECHNICAL_PROMPT,
        default_options={"response_format": TechnicalJudgement},
    )


async def judge_technical(subject: str, documents) -> TechnicalJudgement:
    body = "\n\n".join(f"[{n}] {d.title}\n{d.url}\n{d.text[:4000]}" for n, d in enumerate(documents, 1))
    response = await get_technical_agent().run(
        f"Subject the learner asked for: {subject}\n\nDocuments retrieved for it:\n\n{body}"
    )
    return response.value


# Known-good technical subjects answer question A; the rest answer question B.
BORDERLINE: list[tuple[str, str]] = [
    ("Microsoft Agent Framework", "technical"),
    ("Rust", "technical"),
    ("React", "technical"),
    ("OAuth 2.0", "technical"),
    ("Kubernetes operators", "technical"),
    ("Excel pivot tables", "arguable"),
    ("SEO", "arguable"),
    ("Technical writing", "arguable"),
    ("UX design", "arguable"),
    ("Digital marketing", "arguable"),
    ("Project management", "arguable"),
    ("Statistics", "arguable"),
    ("Photography", "not technical"),
    ("Guitar", "not technical"),
]

print(f"{len(BORDERLINE)} subjects; first-party kinds = {[k.value for k in FIRST_PARTY]}")


14 subjects; first-party kinds = ['specification', 'official_repository', 'first_party_documentation']


In [45]:
BORDER: dict[str, tuple] = globals().get("BORDER", {})

for name, _ in BORDERLINE:
    if name in BORDER:
        continue
    try:
        analysis, documents, trace = await ship_investigate(name)
    except Exception as error:
        print(f"{name:<28} FAILED {type(error).__name__}: {str(error)[:50]}")
        continue
    verdict = await judge_technical(name, documents) if documents else None
    BORDER[name] = (analysis, documents, trace, verdict)
    kinds = sorted({str(item.source_kind) for item in analysis.evidence})
    print(
        f"{name:<28} {str(analysis.identity_status):<22} "
        f"technical={verdict.is_technical if verdict else '-':<6} {kinds}"
    )

print(f"\n{len(BORDER)}/{len(BORDERLINE)} complete")


Microsoft Agent Framework    confirmed              technical=1      ['first_party_documentation', 'official_repository']
Rust                         confirmed              technical=1      ['first_party_documentation']
React                        confirmed              technical=1      ['first_party_documentation', 'official_repository']
OAuth 2.0                    confirmed              technical=1      ['reputable_secondary', 'specification']
Kubernetes operators         confirmed              technical=1      ['first_party_documentation']
Excel pivot tables           confirmed              technical=1      ['first_party_documentation']
SEO                          insufficient_evidence  technical=-      []
Technical writing            confirmed              technical=1      ['reputable_secondary']
UX design                    insufficient_evidence  technical=-      []
Digital marketing            insufficient_evidence  technical=-      []
Project management           insufficien

In [46]:
print("QUESTION A - would 'confirmed needs a first-party source' refuse anything good?\n")
print(f"{'subject':<28} {'status':<22} {'first-party':<12} kinds")
print("-" * 92)

would_refuse: list[str] = []
for name, _ in BORDERLINE:
    if name not in BORDER:
        continue
    analysis, documents, _, _ = BORDER[name]
    kinds = [item.source_kind for item in analysis.evidence]
    has_first_party = any(kind in FIRST_PARTY for kind in kinds)
    confirmed = analysis.identity_status is ShipStatus.CONFIRMED
    print(
        f"{name:<28} {str(analysis.identity_status):<22} "
        f"{('yes' if has_first_party else 'NO'):<12} {sorted({str(k) for k in kinds})}"
    )
    if confirmed and not has_first_party:
        would_refuse.append(name)

print()
if would_refuse:
    print("A first-party rule would REFUSE these correctly-identified subjects:")
    for name in would_refuse:
        print(" -", name)
    print("\nSo the rule costs real subjects. Do not gate on it.")
else:
    print("Every confirmed subject had a first-party source. The rule is free on this sample.")


QUESTION A - would 'confirmed needs a first-party source' refuse anything good?

subject                      status                 first-party  kinds
--------------------------------------------------------------------------------------------
Microsoft Agent Framework    confirmed              yes          ['first_party_documentation', 'official_repository']
Rust                         confirmed              yes          ['first_party_documentation']
React                        confirmed              yes          ['first_party_documentation', 'official_repository']
OAuth 2.0                    confirmed              yes          ['reputable_secondary', 'specification']
Kubernetes operators         confirmed              yes          ['first_party_documentation']
Excel pivot tables           confirmed              yes          ['first_party_documentation']
SEO                          insufficient_evidence  NO           []
Technical writing            confirmed              NO      

In [47]:
print("QUESTION B - where does the technical boundary fall?\n")
print(f"{'subject':<28} {'I expected':<15} {'model says':<12} reason")
print("-" * 110)

disputed: list[str] = []
for name, expectation in BORDERLINE:
    if name not in BORDER:
        continue
    _, _, _, verdict = BORDER[name]
    if verdict is None:
        print(f"{name:<28} {expectation:<15} {'NO DOCS':<12}")
        continue
    said = "technical" if verdict.is_technical else "not technical"
    print(f"{name:<28} {expectation:<15} {said:<12} {verdict.reason[:60]}")
    if expectation != "arguable" and expectation != said:
        disputed.append(f"{name}: I said {expectation}, the model said {said}")

print()
if disputed:
    print("DISAGREEMENT on the cases that were supposed to be clear-cut:")
    for line in disputed:
        print(" -", line)
    print("\nA gate is not safe until the unambiguous ends agree.")
else:
    print("The model and I agree on every clear-cut case. The gate can be trusted at the ends;")
    print("the 'arguable' rows are the product decision, not a correctness one.")


QUESTION B - where does the technical boundary fall?

subject                      I expected      model says   reason
--------------------------------------------------------------------------------------------------------------
Microsoft Agent Framework    technical       technical    True — Microsoft Agent Framework is an open, multi-language 
Rust                         technical       technical    The documents describe Rust as a systems programming languag
React                        technical       technical    The documents describe React as a JavaScript library for bui
OAuth 2.0                    technical       technical    RFC 6749 and oauth.net describe OAuth 2.0 as an IETF-standar
Kubernetes operators         technical       technical    The retrieved documents describe Kubernetes Operators as a K
Excel pivot tables           arguable        technical    Microsoft Support articles describe PivotTables as an intera
SEO                          arguable        NO DOCS    

In [48]:
# The scoreboard said "technical". Read what was actually confirmed, and under what name.
for name in ("Guitar", "Statistics", "Technical writing", "Photography"):
    if name not in BORDER:
        continue
    analysis, documents, trace, _ = BORDER[name]
    print(f"=== asked for: {name}")
    print(f"    status      {analysis.identity_status}")
    print(f"    canonical   {analysis.canonical_name}")
    print(f"    type        {analysis.subject_type}")
    print(f"    description {analysis.description[:150]}")
    for url in trace.fetched_urls:
        print(f"    read        {url}")
    for note in trace.notes:
        print(f"    note        {note}")
    print()


=== asked for: Guitar
    status      confirmed
    canonical   GUITAR
    type        software_framework
    description GUITAR is a GUI testing framework developed at the University of Maryland that provides event-based tools and techniques to automate testing of graphi
    read        https://www.cs.umd.edu/~atif/GUITAR-Web/index.html.old
    note        the plan selected no source; falling back to the top readable hits

=== asked for: Statistics
    status      confirmed
    canonical   statistics
    type        software_library
    description A Python standard-library module that provides functions for calculating mathematical statistics on numeric (Real-valued) data, including measures of 
    read        https://docs.python.org/3/library/statistics.html
    read        https://github.com/python/cpython/blob/main/Lib/statistics.py

=== asked for: Technical writing
    status      confirmed
    canonical   Technical writing
    type        concept_or_practice
    description A s

In [49]:
# The kernel imported the shipped module before the fallback was removed, so reload it.
import importlib

import backend.agents.subject_analysis as subject_module
from backend.services.web_search import search_web as ship_search

importlib.reload(subject_module)
ship_investigate = subject_module.investigate

for name in ("Guitar", "Statistics"):
    analysis, documents, trace = await ship_investigate(name)
    print(f"=== {name}")
    print(f"    status     {analysis.identity_status}")
    print(f"    canonical  {analysis.canonical_name}")
    for url in trace.fetched_urls:
        print(f"    read       {url}")
    for note in trace.notes:
        print(f"    note       {note}")

    # Was the other meaning available and discarded? This is the question that matters.
    print("    --- what the discovery search actually offered:")
    for hit in (await ship_search(name))[:8]:
        print(f"      {hit.title[:70]}")
    print()


=== Guitar
    status     insufficient_evidence
    canonical  None
    note       the plan selected no source, so nothing here settles the identity
    note       no page could be read, so the identity was not put to the model
    --- what the discovery search actually offered:
      GUITAR: an innovative tool for automated testing of GUI-driven ... - U
      GUI Testing FrAmewoRk (GUITAR) - UMD
      GUITAR - A GUI Testing Framework download | SourceForge.net

=== Statistics
    status     insufficient_evidence
    canonical  None
    note       the plan selected no source, so nothing here settles the identity
    note       no page could be read, so the identity was not put to the model
    --- what the discovery search actually offered:
      Statistics · The Julia Language
      statistics — Mathematical statistics functions — Python 3.14.7 ...
      The R Stats Package



In [50]:
# Is the substitution caused by OUR query? search_generic asks:
#   "What is {query}? Cite the official documentation and the project's own site or repository."
# That sentence presumes the subject IS a project. Compare it against a neutral question.
from backend.services.web_search import dedupe as ship_dedupe
from backend.services.web_search import get_search_agent, url_citations

NEUTRAL = "What is {q}? Cite whatever authoritative sources describe it, whatever kind of thing it is."
BIASED = "What is {q}? Cite the official documentation and the project's own site or repository."


async def ask(template: str, query: str):
    response = await get_search_agent().run(template.format(q=query))
    return ship_dedupe(url_citations(response))


for subject in ("Guitar", "Statistics", "Python", "Rust"):
    print(f"=== {subject}")
    for label, template in (("biased ", BIASED), ("neutral", NEUTRAL)):
        hits = await ask(template, subject)
        hosts = [h.url.split("/")[2].removeprefix("www.") for h in hits][:6]
        print(f"  {label}  {hosts}")
    print()


=== Guitar
  biased   ['github.com', 'cs.umd.edu']
  neutral  ['merriam-webster.com', 'dictionary.cambridge.org', 'britannica.com', 'merriam-webster.com']

=== Statistics
  biased   ['docs.python.org', 'docs.julialang.org']
  neutral  ['britannica.com', 'britannica.com', 'web.stanford.edu', 'britannica.com', 'merriam-webster.com']

=== Python
  biased   ['docs.python.org', 'python.org']
  neutral  ['python.org', 'python.org', 'en.wikipedia.org']

=== Rust
  biased   ['doc.rust-lang.org']
  neutral  ['rust-lang.org', 'doc.rust-lang.org']



## 9. The same 14 subjects, with the leading question removed

§8 measured a pipeline whose discovery query asked for *"the project's own site or repository"*.
That question presumed the subject was software and manufactured a namesake whenever it was not,
which is how `Guitar` became a GUI testing framework with genuine first-party documentation.

The question is now neutral. This re-runs the identical subject set so the two are comparable,
and settles the one conclusion that was conditional on the old query: **`is_technical` was
judged dead because it could not see past the substitution. With the substitution gone, it may
work.** Results are kept in `NEUTRAL_BORDER` so `BORDER` stays as the before-picture.


In [51]:
import importlib

import backend.agents.subject_analysis as subject_module
import backend.services.web_search as web_search_module

# subject_analysis binds search_web at import time, so the service reloads first or the agent
# keeps calling the old, leading question. state is NOT reloaded: re-creating the enums would
# break every `is` comparison against SourceKind and ShipStatus.
importlib.reload(web_search_module)
importlib.reload(subject_module)
ship_investigate = subject_module.investigate

NEUTRAL_BORDER: dict[str, tuple] = globals().get("NEUTRAL_BORDER", {})

for name, _ in BORDERLINE:
    if name in NEUTRAL_BORDER:
        continue
    try:
        analysis, documents, trace = await ship_investigate(name)
    except Exception as error:
        print(f"{name:<26} FAILED {type(error).__name__}: {str(error)[:50]}")
        continue
    verdict = await judge_technical(name, documents) if documents else None
    NEUTRAL_BORDER[name] = (analysis, documents, trace, verdict)
    print(
        f"{name:<26} {str(analysis.identity_status):<22} "
        f"technical={str(verdict.is_technical) if verdict else '-':<6} {analysis.canonical_name}"
    )

print(f"\n{len(NEUTRAL_BORDER)}/{len(BORDERLINE)} complete")


Microsoft Agent Framework  confirmed              technical=True   Microsoft Agent Framework
Rust                       confirmed              technical=True   Rust
React                      confirmed              technical=True   React
OAuth 2.0                  confirmed              technical=True   The OAuth 2.0 Authorization Framework
Kubernetes operators       confirmed              technical=True   Operator
Excel pivot tables         confirmed              technical=True   PivotTable
SEO                        confirmed              technical=True   Search Engine Optimization (SEO)
Technical writing          confirmed              technical=False  Professional, Technical Writing
UX design                  confirmed              technical=True   User experience (UX)
Digital marketing          confirmed              technical=True   Digital marketing
Project management         insufficient_evidence  technical=-      None
Statistics                 insufficient_evidence  technical

In [52]:
print("BEFORE = leading query, AFTER = neutral query\n")
print(f"{'subject':<26} {'expected':<14} {'BEFORE':<24} {'AFTER':<24} {'type':<26} tech")
print("-" * 126)

for name, expectation in BORDERLINE:
    before = BORDER.get(name)
    after = NEUTRAL_BORDER.get(name)
    before_text = f"{before[0].identity_status}" if before else "-"
    if before and before[0].canonical_name not in (None, name):
        before_text += f" -> {before[0].canonical_name}"
    if not after:
        print(f"{name:<26} {expectation:<14} {before_text:<24} {'-':<24}")
        continue
    analysis, _, _, verdict = after
    tech = str(verdict.is_technical) if verdict else "-"
    print(
        f"{name:<26} {expectation:<14} {before_text:<24} {str(analysis.identity_status):<24} "
        f"{str(analysis.subject_type):<26} {tech}"
    )

print("\n--- does subject_type separate scope where is_technical did not? ---")
for name, expectation in BORDERLINE:
    after = NEUTRAL_BORDER.get(name)
    if not after or after[0].identity_status is not ShipStatus.CONFIRMED:
        continue
    analysis, _, _, verdict = after
    said = "technical" if verdict and verdict.is_technical else "not technical"
    flag = "  <-- disagrees with expectation" if expectation != "arguable" and expectation != said else ""
    print(f"{name:<26} type={str(analysis.subject_type):<26} is_technical={said}{flag}")


BEFORE = leading query, AFTER = neutral query

subject                    expected       BEFORE                   AFTER                    type                       tech
------------------------------------------------------------------------------------------------------------------------------
Microsoft Agent Framework  technical      confirmed                confirmed                software_framework         True
Rust                       technical      confirmed                confirmed                programming_language       True
React                      technical      confirmed                confirmed                software_library           True
OAuth 2.0                  technical      confirmed                confirmed                protocol_or_specification  True
Kubernetes operators       technical      confirmed -> Operator    confirmed                concept_or_practice        True
Excel pivot tables         arguable       confirmed -> PivotTable  confirmed      